<a href="https://colab.research.google.com/github/amilynestes1028/ds2002-fa26/blob/main/notebooks/04-jsons-apis/Copy_of_2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # Loop up to `tries` times
    for attempt in range(tries):

        # Request with a timeout
        r = requests.get(BASE, params=params, timeout=10)

        # Return DataFrame on success
        if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])

        # Raise immediately if status is not retryable
        if r.status_code not in RETRY_STATUSES:
            r.raise_for_status()
            raise RuntimeError(f"Unexpected status: {r.status_code}")

        # Sleep with increasing delay before retrying
        if attempt < tries - 1:
            time.sleep(2 ** attempt)

    # Raise if every attempt failed
    raise RuntimeError(
        f"Weather request failed after {tries} attempts. "
        f"Last status: {r.status_code}"
    )
    pass

In this part, I created a function called get_weather that retrieves historical weather data from the Open-Meteo API. I added a timeout so the request would not run forever if the server stopped responding. I also included a retry system that attempts the request again if the server is experiencing temporary issues. The delay increases between attempts to avoid overwhelming the API. If there is an error caused by an incorrect request, the function raises an error immediately. This makes the function more reliable and helps prevent incorrect data from being used later.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# Call get_weather once for a range spanning both dates
used_fallback = False

try:
    df = get_weather('2026-09-05', '2026-10-03')

    # Keep only the two dates we care about
    df = df[df['time'].isin(['2026-09-05', '2026-10-03'])]

except (requests.exceptions.RequestException, RuntimeError, ValueError, KeyError) as e:
    used_fallback = True
    print("Weather request failed. Use Part 3 fallback.")
    print("Error:", e)

    df = None

df

,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
28,2026-10-03,22.0,14.7


For this part, I used the get_weather function to collect weather data for September 5 and October 3, 2026. Instead of making two separate API requests, I requested one date range that included both dates. I then filtered the DataFrame to keep only the two dates needed for the comparison. This approach is more efficient because it reduces the number of requests sent to the API. I also included a variable to track whether the request failed and fallback data would be needed.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
# Use fallback if the API request failed
if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = df.copy()
    wx['source'] = 'open-meteo'

# Display results
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


In this section, I created a backup option in case the weather API was unavailable or the request failed. The fallback DataFrame contains sample temperature and precipitation values for the two dates. I used the used_fallback variable to determine whether to use the actual API data or the sample data. I also added a source column to identify where the information came from. This is important because it prevents sample data from being mistaken for actual weather observations and makes the results more transparent.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
# Rename columns
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

# Convert date to datetime
wx['date'] = pd.to_datetime(wx['date'])

# Keep only the required columns
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

# Display results
wx


,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


For this part, I organized the weather data so it could be combined with other datasets, such as daily sales records. I renamed the original API columns to simpler names like date, temp_max, and precip_mm. I also converted the date column into a datetime format so Python could correctly recognize and work with the dates. Finally, I selected only the columns needed for the analysis. This helps keep the dataset organized and makes it easier to merge with other information without confusion.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
# Validate weather data before joining
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']

assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'

assert pd.api.types.is_datetime64_any_dtype(wx['date'])

assert wx['precip_mm'].notna().all(), 'missing precipitation'

assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'

print('weather table is safe to join.')

weather table is safe to join.


In this section, I used assertions to check that the weather data met the requirements before using it in an analysis. I verified that the DataFrame contained the correct column names and exactly two rows. I also checked that the dates were stored in datetime format and that precipitation values were not missing or negative. These checks are important because errors in a dataset can affect the accuracy of later calculations and graphs. By validating the information first, I can catch problems early instead of discovering them after combining multiple datasets.

### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
# Find the wetter and drier dates
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]

# Calculate the difference
difference = wetter['precip_mm'] - drier['precip_mm']

# Print a vendor-friendly sentence
print(f"{wetter['date'].strftime('%B %d, %Y')} was wetter than "
      f"{drier['date'].strftime('%B %d, %Y')} by "
      f"{difference:.1f} mm of rainfall.")

October 03, 2026 was wetter than September 05, 2026 by 12.2 mm of rainfall.


For this part, I compared the precipitation totals from September 5 and October 3, 2026, to determine which date experienced more rainfall. I used Python to identify the date with the highest precipitation and the date with the lowest precipitation. I then calculated the difference between the two rainfall totals in millimeters. Instead of only printing a number, I formatted the result into a sentence that would be easy for a vendor to understand. This makes the analysis more useful because someone reviewing the results can quickly understand the difference in weather conditions.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
from datetime import datetime
from zoneinfo import ZoneInfo

# Cite weather data source
print("Endpoint:", BASE)
print("Coordinates: 38.03, -78.51")
print("Date range: 2026-09-05 to 2026-10-03")
print("Pulled on:", datetime.now(ZoneInfo("America/New_York")).strftime('%Y-%m-%d %H:%M:%S %Z'))
print("Data source:", "fallback sample" if used_fallback else "open-meteo")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled on: 2026-10-11 17:37:50 EDT
Data source: open-meteo


I documented the information used to retrieve the weather data. I included the Open-Meteo API endpoint, the coordinates for the location, and the date range being analyzed. I also printed the current date and time to provide a record of when the citation was generated. Finally, I identified whether the data came from the actual API or the fallback sample. Including this information makes the analysis more transparent and allows someone else to understand where the information originated and how it was collected.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame instead of raising an error, the merge could still run, but the 200 days of sales would have missing weather data. The chart would either be blank, missing points, or misleading if the missing rainfall values were treated as zero. Nobody might catch it because the code could run without an error, making it seem like everything worked. This is why it's important to raise errors and validate the data before using it, so missing information doesn't lead to incorrect conclusions about rainfall and revenue.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
# Get weather for Orlando (one API request)
orlando = get_weather(
    '2026-09-05', '2026-10-03',
    lat=28.54, lon=-81.38
)

# Keep only the two dates
orlando = orlando[
    orlando['time'].isin(['2026-09-05', '2026-10-03'])
].copy()

# Rename columns to match wx
orlando = orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

# Convert dates and add city/source
orlando['date'] = pd.to_datetime(orlando['date'])
orlando['source'] = 'open-meteo'
orlando['city'] = 'Orlando'

# Add city to original weather data
wx['city'] = 'Charlottesville'

# Combine both cities into one long-format DataFrame
weather_both = pd.concat([wx, orlando], ignore_index=True)

# Organize columns
weather_both = weather_both[
    ['date', 'city', 'temp_max', 'precip_mm', 'source']
]

# Display results
weather_both


,date,city,temp_max,precip_mm,source
0,2026-09-05,Charlottesville,29.4,2.5,open-meteo
1,2026-10-03,Charlottesville,22.0,14.7,open-meteo
2,2026-09-05,Orlando,32.1,3.9,open-meteo
3,2026-10-03,Orlando,30.4,14.5,open-meteo
